# Vanguard / WISER 2026 — Presentation Benchmark Suite

This notebook creates an auditable presentation evidence package for five questions:

1. **Correctness:** agreement with exact enumeration and Gurobi on tractable cases.
2. **Financial validity:** simultaneous enforcement of every implemented guardrail.
3. **Quantum contribution:** same-window comparison of exact QUBO search, random sampling, tabu/LNS, XY-QAOA, penalty-QAOA, Aer CPU/GPU, and optional IBM QPU sampling.
4. **Scalability:** factor-native global scaling while the quantum window stays fixed.
5. **Interpretability:** claims generated only from saved CSV/JSON evidence.

## Claims to pursue

| Claim | Minimum evidence |
|---|---|
| Constraint-safe recommendations | Zero independently recomputed hard-constraint breaches |
| Scalable global universe | Repeated factor-native runs through 10,000 assets; 20,000 as a separate stretch result |
| Fixed quantum workload | Fixed window width and approximately flat oracle-call workload as global `n` grows |
| Near-optimal smaller cases | Gurobi incumbent, bound, and reported MIP gap |
| Useful quantum candidates | Better validated objective than warm start or random sampling on the same frozen QUBO |
| Real hardware readiness | IBM job ID, backend, calibration metadata, raw cardinality rate, and validated allocated portfolio |
| Business interpretability | Return, volatility, income, turnover, groups, factors, stress, CVaR, and top-trade explanations |

## Execution order

1. Environment and tests.
2. Tiny exact-certification experiment.
3. Main 100-asset hybrid case.
4. All-constraints gauntlet.
5. Frozen-window method comparison.
6. Width/depth study.
7. Global scaling smoke test.
8. Full 10,000-asset scaling suite.
9. Separate 20,000-asset stretch run.
10. Configure the IBM account once and run the QPU section in the same kernel.

The notebook defaults to 16 numerical threads.

In [1]:
# Run before importing NumPy/SciPy in a fresh kernel.
import os
from pathlib import Path

NUMERICAL_THREADS = 16
for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ.setdefault(variable, str(NUMERICAL_THREADS))

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src" / "vanguard_portfolio").is_dir()
        ):
            return candidate.resolve()
    raise FileNotFoundError("Run from the repository or notebooks/ directory.")

REPO = find_repo_root()
os.chdir(REPO)

# ---------- Master switches ----------
RUN_PYTEST = True
RUN_CONTINUOUS_BACKEND_CROSSCHECK = True
RUN_TINY_CERTIFICATION = True
RUN_MAIN_100_ASSET_CASE = True
RUN_GUROBI_WARM_START_STUDY = True
RUN_BACKTEST_ROBUSTNESS = True
RUN_ALL_CONSTRAINTS_GAUNTLET = True
RUN_CONSTRAINT_ABLATION = True

RUN_FROZEN_WINDOW_BENCHMARK = True
RUN_WIDTH_DEPTH_SWEEP = True
RUN_AER_BACKEND_COMPARISON = True

RUN_SCALING_SMOKE = True
RUN_SCALING_PRESENTATION = True
RUN_100K_STRETCH = True

RUN_IBM_QPU = True
QPU_RESUME = True
QPU_FAIL_FAST = False
RUN_LEGACY_ALTERNATIVE_MODELS = False

OVERWRITE_OUTPUTS = True
BASE_SEED = 20260802
OUTPUT_ROOT = REPO / "results" / "presentation_benchmark_suite"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---------- Classical solver ----------
ALLOCATION_BACKEND = "osqp"
ALLOCATION_TOL = 1.0e-10
ALLOCATION_MAX_ITER = 1_000_000
USE_GUROBI = True
GUROBI_TIME_LIMIT_SMALL = 120.0
GUROBI_TIME_LIMIT_MAIN = 300.0
GUROBI_MIP_GAP = 1.0e-3
BACKTEST_ROBUSTNESS_PATHS = 30
BACKTEST_PERIODS = 120

# ---------- Quantum ----------
LOCAL_QUANTUM_BACKEND_REQUEST = "aer_gpu"
QAOA_DEPTH = 1
QAOA_SHOTS = 4096
QAOA_OPTIMIZER_MAXITER = 80
QAOA_OPTIMIZER_STARTS = 4
QAOA_TOP_CANDIDATES = 128
QAOA_MAX_SUBSPACE_STATES = 400_000
# QAOA_MAX_EDGES = 60
TRANSPILE_OPTIMIZATION_LEVEL = 3

# ---------- Controlled window study ----------
FROZEN_WINDOW_ASSETS = 250
FROZEN_PORTFOLIO_K = 50
FROZEN_WINDOW_WIDTH = 16
# EXACT_QUBO_MAX_STATES = 2_500_000
EXACT_QUBO_TOP_K = 128
RANDOM_CANDIDATE_BUDGET = 128
# WIDTH_SWEEP = [8, 10, 12, 14, 16, 18, 20]
DEPTH_SWEEP_WIDTHS = [10, 12, 14]
DEPTH_SWEEP = [1, 2]
SWEEP_BACKEND = "subspace"
# SWEEP_MAX_SUBSPACE_STATES = 400_000

# Optional stress settings:
WIDTH_SWEEP = [8, 10, 12, 14, 16, 18, 20, 22, 24]
EXACT_QUBO_MAX_STATES = 3_000_000
SWEEP_MAX_SUBSPACE_STATES = 2_500_000
QAOA_MAX_EDGES = 40

# ---------- Global scaling ----------
SCALING_SMOKE_SIZES = [250, 500, 1_000, 2_000]
SCALING_PRESENTATION_SIZES = [250, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 35_000, 50_000, 80_000]
SCALING_REPETITIONS = 1
SCALING_WINDOW_SIZE = 16
SCALING_CARDINALITY = 50
SCALING_GUROBI_MAX_ASSETS = 2_000
SCALING_GUROBI_TIME_LIMIT = 8000

# ---------- IBM Runtime ----------
IBM_BACKEND_NAME = None
QPU_WIDTHS = [8, 10, 12]
QPU_DEPTHS = [1]
QPU_REPETITIONS = 2  # use 3 for final figures
QPU_SHOTS = 4096
QPU_MAX_EDGES = 30
QPU_OPTIMIZER_MAXITER = 100
QPU_OPTIMIZER_STARTS = 5

print("Repository:", REPO)
print("Output root:", OUTPUT_ROOT)

Repository: /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio
Output root: /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio/results/presentation_benchmark_suite


## 1. Environment and branch preflight

In [2]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib import metadata

def command_output(command: list[str], timeout: int = 30) -> str:
    completed = subprocess.run(
        command,
        cwd=REPO,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    return (completed.stdout or completed.stderr).strip()

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "branch": command_output(["git", "branch", "--show-current"]),
    "commit": command_output(["git", "rev-parse", "HEAD"]),
    "status": command_output(["git", "status", "--short"]),
    "cpu": command_output(["bash", "-lc", "lscpu | sed -n '1,25p'"]),
    "memory": command_output(["free", "-h"]),
    "gpu": command_output(["nvidia-smi"]),
}

package_names = [
    "numpy", "scipy", "pandas", "matplotlib", "osqp", "gurobipy",
    "qiskit", "qiskit-aer", "qiskit-aer-gpu",
    "qiskit-aer-gpu-cu11", "qiskit-ibm-runtime",
]
environment["packages"] = {}
for name in package_names:
    try:
        environment["packages"][name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass

aer_devices = ()
aer_advertised_devices = ()
try:
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator

    aer_advertised_devices = tuple(AerSimulator().available_devices())

    def verify_aer_device(device: str) -> None:
        simulator = AerSimulator(method="statevector", device=device)
        circuit = QuantumCircuit(2)
        circuit.h(0)
        circuit.cx(0, 1)
        circuit.measure_all()
        compiled = transpile(circuit, simulator, optimization_level=0)
        result = simulator.run(
            compiled,
            shots=32,
            seed_simulator=BASE_SEED,
        ).result()
        if not bool(getattr(result, "success", True)):
            raise RuntimeError(getattr(result, "status", "unknown status"))

    verified_devices = []
    verify_aer_device("CPU")
    verified_devices.append("CPU")
    if "GPU" in aer_advertised_devices:
        try:
            verify_aer_device("GPU")
            verified_devices.append("GPU")
        except Exception as exc:
            environment["aer_gpu_probe_error"] = (
                f"{type(exc).__name__}: {exc}"
            )
    aer_devices = tuple(verified_devices)
except Exception as exc:
    environment["aer_import_error"] = f"{type(exc).__name__}: {exc}"
environment["aer_advertised_devices"] = list(aer_advertised_devices)
environment["aer_devices"] = list(aer_devices)

installed_aer_distributions = [
    name
    for name in (
        "qiskit-aer",
        "qiskit-aer-gpu",
        "qiskit-aer-gpu-cu11",
    )
    if name in environment["packages"]
]
environment["installed_aer_distributions"] = installed_aer_distributions

if RUN_IBM_QPU:
    expected_versions = {
        "qiskit": "2.5.1",
        "qiskit-ibm-runtime": "0.48.0",
    }
    version_mismatches = {
        name: {
            "expected": expected,
            "installed": environment["packages"].get(name),
        }
        for name, expected in expected_versions.items()
        if environment["packages"].get(name) != expected
    }
    if len(installed_aer_distributions) != 1:
        raise RuntimeError(
            "The QPU notebook requires exactly one Aer distribution, but "
            f"found {installed_aer_distributions}. Run "
            f"{sys.executable} scripts/install_environment.py, then restart "
            "this kernel."
        )
    if version_mismatches:
        raise RuntimeError(
            "The notebook kernel is not using the pinned unified quantum "
            f"stack: {version_mismatches}. Interpreter: {sys.executable}"
        )
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    except Exception as exc:
        raise RuntimeError(
            "IBM Runtime cannot import in this notebook kernel. Run "
            f"{sys.executable} -m pip install -e '.[full]' from the repository, "
            "restart the kernel, and rerun the preflight."
        ) from exc
    environment["ibm_runtime_import"] = "ok"

gurobi_available = False
try:
    import gurobipy as gp
    probe = gp.Model()
    probe.Params.OutputFlag = 0
    gurobi_available = True
    environment["gurobi_version"] = ".".join(map(str, gp.gurobi.version()))
except Exception as exc:
    environment["gurobi_error"] = f"{type(exc).__name__}: {exc}"
environment["gurobi_available"] = gurobi_available

assert environment["branch"] == "research/corrections", (
    f"Expected research/corrections; found {environment['branch']!r}"
)

LOCAL_QUANTUM_BACKEND = (
    "aer_gpu"
    if LOCAL_QUANTUM_BACKEND_REQUEST == "aer_gpu" and "GPU" in aer_devices
    else "subspace"
)
if LOCAL_QUANTUM_BACKEND_REQUEST == "aer_gpu" and LOCAL_QUANTUM_BACKEND != "aer_gpu":
    print("WARNING: Aer GPU unavailable; local quantum tests use subspace.")

(OUTPUT_ROOT / "environment.json").write_text(
    json.dumps(environment, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(environment, indent=2))
print("Resolved local quantum backend:", LOCAL_QUANTUM_BACKEND)
print("Gurobi enabled:", bool(USE_GUROBI and gurobi_available))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2845661
Academic license 2845661 - for non-commercial use only - registered to ap___@usf.edu
{
  "timestamp_utc": "2026-08-05T17:43:24.687818+00:00",
  "python": "3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]",
  "platform": "Linux-6.1.0-34-amd64-x86_64-with-glibc2.36",
  "branch": "research/corrections",
  "commit": "4f87839890e0cb8cce7706087c1c7391f74fa3a8",
  "status": "M notebooks/Vanguard_Presentation_Benchmark_Suite.ipynb\n M src/vanguard_portfolio/allocation.py",
  "cpu": "Architecture:                         x86_64\nCPU op-mode(s):                       32-bit, 64-bit\nAddress sizes:                        48 bits physical, 48 bits virtual\nByte Order:                           Little Endian\nCPU(s):                               32\nOn-line CPU(s) list:                  0-31\nVendor ID:                            AuthenticAMD\nModel name:                           AMD Ryzen 9 5950X 16-Core

## 2. Imports and helpers

In [3]:
import math
import shutil
import time
from dataclasses import replace
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from vanguard_portfolio.allocation import (
    AllocationOracle,
    find_feasible_initial_support,
    solve_relaxation,
)
from vanguard_portfolio.classical_continuous import solve_continuous
from vanguard_portfolio.classical_discrete import solve_cardinality_gurobi
from vanguard_portfolio.data_generation import (
    generate_backtest_returns,
    generate_factor_universe,
    generate_return_scenarios,
    generate_synthetic_universe,
)
from vanguard_portfolio.hybrid import HybridConfig, HybridRun, run_hybrid_optimizer
from vanguard_portfolio.presentation import write_hybrid_artifacts
from vanguard_portfolio.quantum_solver import (
    XYQAOAConfig,
    _basis_energies,
    _fixed_weight_basis,
    solve_xy_qaoa,
)
from vanguard_portfolio.qubo_builder import build_window_qubo
from vanguard_portfolio.schemas import PortfolioConstraints, PortfolioProblem, Preferences
from vanguard_portfolio.topology import market_communities
from vanguard_portfolio.validation import validate_weights
from vanguard_portfolio.window_search import (
    construct_change_window,
    current_window_bits,
    evaluate_bitstrings,
    tabu_window_search,
)

PREFERENCES = Preferences(
    lambda_return=1.0,
    lambda_risk=5.0,
    lambda_income=0.5,
    lambda_cost=1.0,
)

def clean_directory(path: Path) -> None:
    if path.exists() and OVERWRITE_OUTPUTS:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def result_table(run: HybridRun) -> pd.DataFrame:
    frame = pd.DataFrame(run.summary_records())
    columns = [
        "method", "stage", "iteration", "objective", "runtime_seconds",
        "feasible", "breaches", "max_violation", "support_size",
        "expected_return", "volatility", "income", "turnover",
        "best_bound", "reported_mip_gap",
    ]
    return frame[[column for column in columns if column in frame.columns]]

def run_and_archive(
    label: str,
    problem: PortfolioProblem,
    constraints: PortfolioConstraints,
    config: HybridConfig,
    *,
    backtest_periods: int = 120,
    backtest_seed: int = BASE_SEED + 1,
) -> tuple[HybridRun, pd.DataFrame, Path]:
    output = OUTPUT_ROOT / label
    clean_directory(output)
    total_start = time.perf_counter()
    run = run_hybrid_optimizer(problem, PREFERENCES, constraints, config)
    realized = generate_backtest_returns(
        problem,
        periods=backtest_periods,
        periods_per_year=12,
        seed=backtest_seed,
    )
    write_hybrid_artifacts(run, output, realized_returns=realized)
    frame = result_table(run)
    frame.to_csv(output / "notebook_method_summary.csv", index=False)
    (output / "notebook_wall_seconds.txt").write_text(
        f"{time.perf_counter() - total_start:.9f}\n",
        encoding="utf-8",
    )
    print(
        f"{label}: best={run.best.method}, objective={run.best.objective:.10g}, "
        f"breaches={run.best.breaches}, runtime={run.runtime:.3f}s"
    )
    display(frame)
    return run, frame, output

def empirical_cvar(
    scenario_returns: np.ndarray,
    weights: np.ndarray,
    alpha: float = 0.95,
) -> float:
    losses = -(np.asarray(scenario_returns) @ np.asarray(weights))
    tail_size = max(1, int(math.ceil((1.0 - alpha) * losses.size)))
    return float(np.mean(np.partition(losses, losses.size - tail_size)[-tail_size:]))

def support(weights: np.ndarray, tolerance: float = 1.0e-8) -> set[int]:
    return set(np.flatnonzero(np.asarray(weights) > tolerance).tolist())

def support_jaccard(left: np.ndarray, right: np.ndarray) -> float:
    a, b = support(left), support(right)
    return len(a & b) / max(len(a | b), 1)

def best_by_method(run: HybridRun) -> dict[str, object]:
    selected = {}
    for result in run.all_results():
        if not (result.success and result.feasible):
            continue
        previous = selected.get(result.method)
        if previous is None or result.objective < previous.objective:
            selected[result.method] = result
    return selected

def plot_top_trades(run: HybridRun, output: Path, count: int = 10) -> Path:
    change = run.best.weights - run.problem.w0
    buys = np.argsort(change)[-count:][::-1]
    sells = np.argsort(change)[:count]
    indices = np.concatenate([sells, buys])
    figure, axis = plt.subplots(figsize=(12, 5))
    axis.bar(np.arange(len(indices)), change[indices])
    axis.axhline(0.0, linewidth=1.0)
    axis.set_xticks(np.arange(len(indices)))
    axis.set_xticklabels(
        [run.problem.asset_names[index] for index in indices],
        rotation=55,
        ha="right",
    )
    axis.set_ylabel("Recommended weight minus current weight")
    axis.set_title("Largest recommended sales and purchases")
    figure.tight_layout()
    path = output / "top_trades.png"
    figure.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    return path

def plot_support_overlap(run: HybridRun, output: Path) -> Path:
    methods = best_by_method(run)
    names = list(methods)
    matrix = np.empty((len(names), len(names)))
    for i, left in enumerate(names):
        for j, right in enumerate(names):
            matrix[i, j] = support_jaccard(
                methods[left].weights,
                methods[right].weights,
            )
    figure, axis = plt.subplots(figsize=(8, 7))
    image = axis.imshow(matrix, vmin=0.0, vmax=1.0)
    axis.set_xticks(range(len(names)))
    axis.set_yticks(range(len(names)))
    axis.set_xticklabels(names, rotation=55, ha="right")
    axis.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            axis.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
    axis.set_title("Support overlap (Jaccard similarity)")
    figure.colorbar(image, ax=axis)
    figure.tight_layout()
    path = output / "support_overlap.png"
    figure.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    return path

## 3. Test suite

In [4]:
if RUN_PYTEST:
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=REPO,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError("The test suite failed; stop before generating claims.")

...............................................................          [100%]
63 passed in 6.20s


## 3A. Continuous-backend cross-check

This small convex case compares every canonical continuous backend that is installed:

- SciPy/SLSQP;
- OSQP;
- CVXPY/Clarabel;
- Gurobi QP.

The purpose is numerical cross-validation, not a claim that every backend should be used in the production pipeline. The portfolio model and objective are identical across rows.

In [5]:
continuous_backend_crosscheck = None

if RUN_CONTINUOUS_BACKEND_CROSSCHECK:
    cross_problem = generate_factor_universe(
        n_assets=100,
        n_groups=8,
        n_factors=6,
        seed=BASE_SEED + 11,
        current_cardinality=None,
        materialize_covariance=False,
    )
    cross_problem.target_return = None
    cross_problem.max_turnover = None

    backend_specs = [
        ("scipy", {"tol": 1.0e-10, "max_iter": 5_000}),
        (
            "osqp",
            {
                "tol": ALLOCATION_TOL,
                "max_iter": ALLOCATION_MAX_ITER,
            },
        ),
        ("cvxpy:CLARABEL", {"solver_options": {}}),
    ]
    if USE_GUROBI and gurobi_available:
        backend_specs.append(
            (
                "gurobi",
                {
                    "threads": NUMERICAL_THREADS,
                    "seed": BASE_SEED,
                    "method": 2,
                    "bar_conv_tol": 1.0e-10,
                    "output": False,
                },
            )
        )

    rows = []
    weights_by_backend = {}
    for backend, options in backend_specs:
        try:
            result = solve_continuous(
                cross_problem,
                PREFERENCES,
                backend=backend,
                **options,
            )
            rows.append(
                {
                    "backend": backend,
                    "method": result.method,
                    "status": result.status,
                    "success": result.success,
                    "feasible": result.feasible,
                    "objective": result.objective,
                    "runtime_seconds": result.runtime,
                    "breaches": result.breaches,
                    "max_violation": result.max_violation,
                    "iterations": result.metadata.get("iterations"),
                }
            )
            if result.success and result.feasible:
                weights_by_backend[backend] = result.weights.copy()
        except Exception as exc:
            rows.append(
                {
                    "backend": backend,
                    "status": "unavailable_or_failed",
                    "success": False,
                    "feasible": False,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    continuous_backend_crosscheck = pd.DataFrame(rows)
    display(continuous_backend_crosscheck)
    continuous_backend_crosscheck.to_csv(
        OUTPUT_ROOT / "continuous_backend_crosscheck.csv",
        index=False,
    )

    successful_rows = continuous_backend_crosscheck[
        continuous_backend_crosscheck["success"] == True
    ]
    if len(successful_rows) >= 2:
        objective_spread = (
            successful_rows["objective"].max()
            - successful_rows["objective"].min()
        )
        pairwise_weight_differences = []
        names = list(weights_by_backend)
        for left_index, left in enumerate(names):
            for right in names[left_index + 1 :]:
                pairwise_weight_differences.append(
                    {
                        "left": left,
                        "right": right,
                        "maximum_absolute_weight_difference": float(
                            np.max(
                                np.abs(
                                    weights_by_backend[left]
                                    - weights_by_backend[right]
                                )
                            )
                        ),
                    }
                )
        print("Objective spread:", objective_spread)
        display(pd.DataFrame(pairwise_weight_differences))

,backend,method,status,success,feasible,objective,runtime_seconds,breaches,max_violation,iterations
0,scipy,scipy_slsqp,Optimization terminated successfully,True,True,-0.049141,1.376480,0,2.220446e-16,61.0
1,osqp,osqp,solved,True,True,-0.049141,0.003856,0,1.568746e-10,300.0
2,cvxpy:CLARABEL,cvxpy_clarabel,optimal,True,True,-0.049141,0.010097,0,2.220446e-16,10.0
3,gurobi,gurobi_qp,optimal,True,True,-0.049141,0.021490,0,7.257817e-11,0.0


Objective spread: 1.4542636886383242e-09


,left,right,maximum_absolute_weight_difference
0,scipy,osqp,0.000045
1,scipy,cvxpy:CLARABEL,0.000168
2,scipy,gurobi,0.000055
3,osqp,cvxpy:CLARABEL,0.000160
4,osqp,gurobi,0.000013
5,cvxpy:CLARABEL,gurobi,0.000146


## 4. Experiment A — tiny exact correctness and certification

This tractable case exercises exact window enumeration, XY-QAOA, penalty-QAOA, the exact continuous allocation oracle, independent validation, and optional Gurobi certification.

In [6]:
tiny_run = tiny_frame = tiny_output = None

if RUN_TINY_CERTIFICATION:
    tiny_problem = generate_synthetic_universe()
    tiny_problem.target_return = None
    tiny_problem.max_turnover = 0.80

    tiny_constraints = PortfolioConstraints(
        exact_cardinality=4,
        minimum_active_weight=0.05,
        maximum_weights=np.minimum(tiny_problem.upper, 0.50),
    ).validate_for(tiny_problem)

    tiny_config = HybridConfig(
        iterations=1,
        window_size=6,
        held_fraction=0.50,
        allocation_backend="scipy",
        allocation_options={"tol": 1.0e-10, "max_iter": 5_000},
        initial_trials=300,
        initial_milp_time_limit=20.0,
        classical_tabu_iterations=60,
        classical_tabu_tenure=7,
        classical_oracle_candidates=4,
        enumerate_windows_up_to=100_000,
        run_quantum=True,
        quantum=XYQAOAConfig(
            depth=1,
            shots=QAOA_SHOTS,
            optimizer_maxiter=120,
            optimizer_starts=6,
            seed=BASE_SEED,
            initial_state="warm",
            mixer="ring",
            backend="subspace",
            maximum_subspace_states=400_000,
            top_candidates=128,
            transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
        ),
        run_penalty_qaoa=True,
        penalty_multiplier=10.0,
        use_topology=False,
        maximum_quantum_edges=None,
        run_gurobi_reference=bool(USE_GUROBI and gurobi_available),
        gurobi_time_limit=GUROBI_TIME_LIMIT_SMALL,
        gurobi_mip_gap=1.0e-6,
        seed=BASE_SEED,
    )

    tiny_run, tiny_frame, tiny_output = run_and_archive(
        "01_tiny_exact_certification",
        tiny_problem,
        tiny_constraints,
        tiny_config,
        backtest_periods=60,
    )

    exact_rows = tiny_frame[
        tiny_frame["method"].isin(
            ["classical_enumeration", "gurobi_cardinality_miqp"]
        )
    ]
    if len(exact_rows) >= 2:
        print(
            "Enumeration/Gurobi objective spread:",
            exact_rows["objective"].max() - exact_rows["objective"].min(),
        )

01_tiny_exact_certification: best=feasible_initial_portfolio, objective=-0.03681145, breaches=0, runtime=0.116s


,method,stage,iteration,objective,runtime_seconds,feasible,breaches,max_violation,support_size,expected_return,volatility,income,turnover,best_bound,reported_mip_gap
0,scipy_slsqp,,,-0.038073,0.008997,True,0,5.551115e-17,5,0.04223,0.061072,0.030111,0.5,,
1,feasible_initial_portfolio,initialization,,-0.036811,0.006853,True,0,1.110223e-16,4,0.04350,0.065404,0.031000,0.7,,
2,classical_enumeration,window_search,0,-0.036811,0.076217,True,0,1.110223e-16,4,0.04350,0.065404,0.031000,0.7,,
3,penalty_qaoa_statevector,window_search,0,-0.036811,0.005362,True,0,1.110223e-16,4,0.04350,0.065404,0.031000,0.7,,
4,gurobi_cardinality_miqp,global_certification,1,-0.036811,0.008461,True,0,1.110223e-16,4,0.04350,0.065404,0.031000,0.7,-0.036811,0.0


Enumeration/Gurobi objective spread: 0.0


## 5. Experiment B — main 100-asset canonical hybrid case

This presentation-friendly case uses the complete canonical pipeline: factor-QP relaxation, exact-\(K\) initialization, topology-aware windows, tabu/LNS, XY-QAOA, first-window penalty-QAOA, exact reallocation, validation, optional Gurobi certification, and backtesting.

In [7]:
main_run = main_frame = main_output = None

if RUN_MAIN_100_ASSET_CASE:
    main_problem = generate_factor_universe(
        n_assets=100,
        n_groups=8,
        n_factors=6,
        seed=BASE_SEED,
        current_cardinality=20,
        materialize_covariance=False,
    )
    main_problem.target_return = None
    main_problem.max_turnover = 0.40

    main_constraints = PortfolioConstraints(
        exact_cardinality=20,
        minimum_active_weight=0.01,
        maximum_weights=np.full(main_problem.n, 0.10),
    ).validate_for(main_problem)

    main_config = HybridConfig(
        iterations=2,
        window_size=16,
        held_fraction=0.45,
        allocation_backend=ALLOCATION_BACKEND,
        allocation_options={
            "tol": ALLOCATION_TOL,
            "max_iter": ALLOCATION_MAX_ITER,
        },
        initial_trials=400,
        initial_milp_time_limit=30.0,
        classical_tabu_iterations=80,
        classical_tabu_tenure=8,
        classical_oracle_candidates=4,
        enumerate_windows_up_to=5_000,
        run_quantum=True,
        quantum=XYQAOAConfig(
            depth=QAOA_DEPTH,
            shots=QAOA_SHOTS,
            optimizer_maxiter=QAOA_OPTIMIZER_MAXITER,
            optimizer_starts=QAOA_OPTIMIZER_STARTS,
            seed=BASE_SEED,
            initial_state="warm",
            mixer="ring",
            backend=LOCAL_QUANTUM_BACKEND,
            maximum_subspace_states=QAOA_MAX_SUBSPACE_STATES,
            top_candidates=QAOA_TOP_CANDIDATES,
            transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
        ),
        run_penalty_qaoa=True,
        penalty_multiplier=10.0,
        use_topology=True,
        maximum_quantum_edges=QAOA_MAX_EDGES,
        run_gurobi_reference=bool(USE_GUROBI and gurobi_available),
        gurobi_time_limit=GUROBI_TIME_LIMIT_MAIN,
        gurobi_mip_gap=GUROBI_MIP_GAP,
        seed=BASE_SEED,
    )

    main_run, main_frame, main_output = run_and_archive(
        "02_main_100_asset_case",
        main_problem,
        main_constraints,
        main_config,
    )
    plot_top_trades(main_run, main_output)
    plot_support_overlap(main_run, main_output)

02_main_100_asset_case: best=gurobi_cardinality_miqp, objective=-0.03841471459, breaches=0, runtime=2.246s


,method,stage,iteration,objective,runtime_seconds,feasible,breaches,max_violation,support_size,expected_return,volatility,income,turnover,best_bound,reported_mip_gap
0,osqp,,,-0.038416,0.009649,True,0,2.056619e-10,20,0.061667,0.082019,0.022144,0.4,,
1,feasible_initial_portfolio,initialization,,-0.034688,0.009019,True,0,5.969409e-10,20,0.060506,0.083795,0.019962,0.4,,
2,classical_tabu_lns,window_search,0,-0.036845,0.348149,True,0,1.188096e-09,20,0.059421,0.080226,0.020480,0.4,,
3,xy_qaoa_subspace,window_search,0,-0.036751,0.172783,True,0,2.944806e-11,20,0.060866,0.082498,0.021127,0.4,,
4,penalty_qaoa_statevector,window_search,0,-0.037317,1.351507,True,0,7.696810e-11,20,0.061711,0.082929,0.021620,0.4,,
5,classical_tabu_lns,window_search,1,-0.037317,0.231757,True,0,7.696810e-11,20,0.061711,0.082929,0.021620,0.4,,
6,gurobi_cardinality_miqp,global_certification,2,-0.038415,0.021299,True,0,2.220446e-16,20,0.061726,0.082073,0.022117,0.4,-0.038415,0.0


## 5A. Fair Gurobi cold-start versus warm-start study

The normal hybrid pipeline calls Gurobi **after** the local search and supplies the hybrid incumbent as a MIP start. That is useful for certification, but it is not a fair independent speed race.

This optional study compares three otherwise identical Gurobi solves:

1. cold start;
2. warm start from the feasible initializer;
3. warm start from the best hybrid result before Gurobi.

In [8]:
gurobi_start_study = None

if (
    RUN_GUROBI_WARM_START_STUDY
    and main_run is not None
    and USE_GUROBI
    and gurobi_available
):
    hybrid_candidates = [
        main_run.initial,
        *[
            result
            for result in main_run.results
            if (
                result.method != "gurobi_cardinality_miqp"
                and result.success
                and result.feasible
            )
        ],
    ]
    hybrid_best_before_gurobi = min(
        hybrid_candidates,
        key=lambda result: result.objective,
    )

    starts = {
        "cold_start": None,
        "valid_initial_start": main_run.initial.weights,
        "hybrid_incumbent_start": hybrid_best_before_gurobi.weights,
    }
    rows = []
    for label, warm_start in starts.items():
        result = solve_cardinality_gurobi(
            main_problem,
            PREFERENCES,
            main_constraints,
            warm_start=warm_start,
            time_limit=GUROBI_TIME_LIMIT_MAIN,
            mip_gap=GUROBI_MIP_GAP,
            threads=NUMERICAL_THREADS,
            seed=BASE_SEED,
            output=False,
        )
        rows.append(
            {
                "start": label,
                "status": result.status,
                "success": result.success,
                "optimal": result.optimal,
                "feasible": result.feasible,
                "objective": result.objective,
                "best_bound": result.metadata.get("best_bound"),
                "reported_mip_gap": result.metadata.get(
                    "reported_mip_gap"
                ),
                "model_build_seconds": result.metadata.get(
                    "model_build_seconds"
                ),
                "solve_seconds": result.metadata.get("solve_seconds"),
                "total_seconds": result.runtime,
                "nodes": result.metadata.get("nodes"),
                "breaches": result.breaches,
            }
        )

    gurobi_start_study = pd.DataFrame(rows)
    display(gurobi_start_study)
    gurobi_start_study.to_csv(
        main_output / "gurobi_cold_vs_warm.csv",
        index=False,
    )

    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].bar(
        gurobi_start_study["start"],
        gurobi_start_study["total_seconds"],
    )
    axes[0].set_ylabel("Wall-clock seconds")
    axes[0].set_title("Gurobi runtime by MIP start")
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(
        gurobi_start_study["start"],
        gurobi_start_study["reported_mip_gap"],
    )
    axes[1].set_ylabel("Reported MIP gap")
    axes[1].set_title("Certificate quality at the time limit")
    axes[1].tick_params(axis="x", rotation=25)

    figure.tight_layout()
    figure.savefig(
        main_output / "gurobi_cold_vs_warm.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

,start,status,success,optimal,feasible,objective,best_bound,reported_mip_gap,model_build_seconds,solve_seconds,total_seconds,nodes,breaches
0,cold_start,optimal,True,True,True,-0.038415,-0.038415,0.0,0.014826,0.020073,0.035008,1.0,0
1,valid_initial_start,optimal,True,True,True,-0.038415,-0.038415,0.0,0.015690,0.018489,0.034250,1.0,0
2,hybrid_incumbent_start,optimal,True,True,True,-0.038415,-0.038415,0.0,0.012030,0.014552,0.026650,1.0,0


## 5B. Multi-seed out-of-sample robustness

A single synthetic wealth path is illustrative but cannot support a robustness claim. This inexpensive study evaluates the already-computed portfolios on many independent factor-model paths and reports distributions of:

- terminal wealth;
- annualized return;
- annualized volatility;
- return-to-volatility ratio;
- maximum drawdown;
- empirical 95% CVaR.

No optimizer is rerun, so this isolates portfolio behavior from optimization randomness.

In [9]:
backtest_robustness = None
backtest_robustness_summary = None

if RUN_BACKTEST_ROBUSTNESS and main_run is not None:
    methods = {
        name: result
        for name, result in best_by_method(main_run).items()
        if result.model_type != "continuous_relaxation"
    }

    rows = []
    for path_index in range(BACKTEST_ROBUSTNESS_PATHS):
        realized = generate_backtest_returns(
            main_problem,
            periods=BACKTEST_PERIODS,
            periods_per_year=12,
            seed=BASE_SEED + 10_000 + path_index,
        )
        for method, result in methods.items():
            portfolio_returns = realized @ result.weights
            wealth = np.cumprod(1.0 + portfolio_returns)
            running_peak = np.maximum.accumulate(
                np.concatenate([[1.0], wealth])
            )[1:]
            drawdown = 1.0 - wealth / np.maximum(
                running_peak,
                1.0e-15,
            )
            annual_return = float(
                np.mean(portfolio_returns) * 12.0
            )
            annual_volatility = float(
                np.std(portfolio_returns, ddof=1) * np.sqrt(12.0)
            )
            rows.append(
                {
                    "path": path_index,
                    "seed": BASE_SEED + 10_000 + path_index,
                    "method": method,
                    "terminal_wealth": float(wealth[-1]),
                    "annualized_return": annual_return,
                    "annualized_volatility": annual_volatility,
                    "return_to_volatility": (
                        annual_return
                        / max(annual_volatility, 1.0e-15)
                    ),
                    "maximum_drawdown": float(np.max(drawdown)),
                    "period_cvar_95": empirical_cvar(
                        realized,
                        result.weights,
                        alpha=0.95,
                    ),
                }
            )

    backtest_robustness = pd.DataFrame(rows)
    metrics = [
        "terminal_wealth",
        "annualized_return",
        "annualized_volatility",
        "return_to_volatility",
        "maximum_drawdown",
        "period_cvar_95",
    ]
    summary_rows = []
    for method, group in backtest_robustness.groupby("method"):
        for metric in metrics:
            values = group[metric].to_numpy()
            summary_rows.append(
                {
                    "method": method,
                    "metric": metric,
                    "median": float(np.median(values)),
                    "q10": float(np.quantile(values, 0.10)),
                    "q90": float(np.quantile(values, 0.90)),
                    "mean": float(np.mean(values)),
                    "std": float(np.std(values, ddof=1)),
                }
            )
    backtest_robustness_summary = pd.DataFrame(summary_rows)
    display(backtest_robustness_summary)

    backtest_robustness.to_csv(
        main_output / "backtest_robustness_paths.csv",
        index=False,
    )
    backtest_robustness_summary.to_csv(
        main_output / "backtest_robustness_summary.csv",
        index=False,
    )

    method_order = list(methods)
    figure, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].boxplot(
        [
            backtest_robustness.loc[
                backtest_robustness["method"] == method,
                "terminal_wealth",
            ].to_numpy()
            for method in method_order
        ],
        tick_labels=method_order,  # Fix applied here
    )
    axes[0].set_title("Terminal wealth across independent paths")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].boxplot(
        [
            backtest_robustness.loc[
                backtest_robustness["method"] == method,
                "maximum_drawdown",
            ].to_numpy()
            for method in method_order
        ],
        tick_labels=method_order,  # Fix applied here
    )
    axes[1].set_title("Maximum drawdown across independent paths")
    axes[1].tick_params(axis="x", rotation=45)


    figure.tight_layout()
    figure.savefig(
        main_output / "backtest_robustness.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

,method,metric,median,q10,q90,mean,std
0,classical_tabu_lns,terminal_wealth,1.791235,1.356482,2.435244,1.804718,0.467590
1,classical_tabu_lns,annualized_return,0.062035,0.033931,0.093006,0.059440,0.026168
2,classical_tabu_lns,annualized_volatility,0.082784,0.080177,0.090108,0.084041,0.003940
3,classical_tabu_lns,return_to_volatility,0.691766,0.403229,1.119658,0.709073,0.320523
4,classical_tabu_lns,maximum_drawdown,0.144153,0.097089,0.218113,0.149696,0.046367
5,classical_tabu_lns,period_cvar_95,0.041231,0.037620,0.048069,0.042290,0.004451
6,feasible_initial_portfolio,terminal_wealth,1.735806,1.301503,2.402762,1.763866,0.457879
7,feasible_initial_portfolio,annualized_return,0.058480,0.029889,0.091810,0.057196,0.026240
8,feasible_initial_portfolio,annualized_volatility,0.083993,0.080657,0.090771,0.084905,0.004189
9,feasible_initial_portfolio,return_to_volatility,0.708319,0.354794,1.131392,0.675511,0.317028


## 6. Experiment C — all-constraints gauntlet

Small all-constraints certification case

The gauntlet activates:

- budget and group bounds;
- exact cardinality;
- minimum and maximum active weights;
- eligibility and mandatory assets;
- target return;
- turnover cap;
- minimum income;
- factor exposure bands;
- stress-scenario floors;
- empirical CVaR.

Every advanced limit is calibrated around the known-feasible current portfolio, which is independently validated before optimization.

In [10]:
def build_constraint_gauntlet(
    *,
    n_assets: int = 100,
    cardinality: int = 20,
    seed: int = BASE_SEED + 101,
    scenario_count: int = 500,
    stress_count: int = 5,
    cvar_alpha: float = 0.95,
) -> tuple[PortfolioProblem, PortfolioConstraints, dict[str, float]]:
    problem = generate_factor_universe(
        n_assets=n_assets,
        n_groups=8,
        n_factors=6,
        seed=seed,
        current_cardinality=cardinality,
        materialize_covariance=False,
    )
    anchor = problem.w0.copy()
    anchor_support = np.flatnonzero(anchor > 1.0e-8)

    anchor_return = float(problem.mu @ anchor)
    anchor_income = float(problem.y @ anchor)
    problem.target_return = anchor_return - max(0.0010, 0.03 * abs(anchor_return))
    problem.max_turnover = 0.40

    rng = np.random.default_rng(seed + 1)
    nonheld = np.flatnonzero(anchor <= 1.0e-8)
    excluded_count = min(max(5, n_assets // 10), max(len(nonheld) - cardinality, 0))
    excluded = (
        set(rng.choice(nonheld, size=excluded_count, replace=False).tolist())
        if excluded_count
        else set()
    )
    eligible = tuple(index for index in range(n_assets) if index not in excluded)
    mandatory = tuple(int(index) for index in anchor_support[:3])

    factor_anchor = problem.factor_loadings.T @ anchor
    factor_band = np.maximum(0.005, 0.10 * np.maximum(np.abs(factor_anchor), 0.01))

    scenarios = generate_return_scenarios(
        problem,
        n_scenarios=scenario_count,
        seed=seed + 2,
    )
    anchor_scenario_returns = scenarios @ anchor
    stress_indices = np.argsort(anchor_scenario_returns)[:stress_count]
    stress_scenarios = scenarios[stress_indices]
    stress_floors = stress_scenarios @ anchor - 0.0025

    anchor_cvar = empirical_cvar(scenarios, anchor, cvar_alpha)
    maximum_cvar = anchor_cvar + max(0.0025, 0.05 * abs(anchor_cvar))

    constraints = PortfolioConstraints(
        exact_cardinality=cardinality,
        minimum_active_weight=0.01,
        eligible_assets=eligible,
        mandatory_assets=mandatory,
        minimum_income=anchor_income - max(0.0010, 0.03 * abs(anchor_income)),
        maximum_weights=np.full(n_assets, 0.10),
        factor_lower=factor_anchor - factor_band,
        factor_upper=factor_anchor + factor_band,
        stress_scenarios=stress_scenarios,
        stress_floors=stress_floors,
        scenario_returns=scenarios,
        maximum_cvar=maximum_cvar,
        cvar_alpha=cvar_alpha,
    ).validate_for(problem)

    anchor_report = validate_weights(anchor, problem, constraints=constraints)
    if not anchor_report.feasible:
        failed = [
            check.name
            for check in anchor_report.checks
            if check.violation > 1.0e-7
        ]
        raise RuntimeError(f"Calibrated anchor is not feasible: {failed}")

    calibration = {
        "anchor_return": anchor_return,
        "target_return": float(problem.target_return),
        "anchor_income": anchor_income,
        "minimum_income": float(constraints.minimum_income),
        "anchor_cvar": anchor_cvar,
        "maximum_cvar": maximum_cvar,
        "eligible_assets": len(eligible),
        "mandatory_assets": len(mandatory),
        "stress_scenarios": stress_count,
        "cvar_scenarios": scenario_count,
    }
    return problem, constraints, calibration

gauntlet_run = gauntlet_frame = gauntlet_output = None
gauntlet_calibration = None

if RUN_ALL_CONSTRAINTS_GAUNTLET:
    gauntlet_problem, gauntlet_constraints, gauntlet_calibration = (
        build_constraint_gauntlet()
    )
    display(pd.DataFrame([gauntlet_calibration]).T.rename(columns={0: "value"}))

    gauntlet_config = HybridConfig(
        iterations=2,
        window_size=12,
        held_fraction=0.42,
        allocation_backend="scipy",
        allocation_options={"tol": 1.0e-9, "max_iter": 8_000},
        initial_trials=500,
        initial_milp_time_limit=60.0,
        classical_tabu_iterations=80,
        classical_tabu_tenure=8,
        classical_oracle_candidates=4,
        enumerate_windows_up_to=5_000,
        run_quantum=True,
        quantum=XYQAOAConfig(
            depth=1,
            shots=QAOA_SHOTS,
            optimizer_maxiter=100,
            optimizer_starts=5,
            seed=BASE_SEED + 101,
            initial_state="warm",
            mixer="ring",
            backend=LOCAL_QUANTUM_BACKEND,
            maximum_subspace_states=400_000,
            top_candidates=128,
            transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
        ),
        run_penalty_qaoa=True,
        penalty_multiplier=10.0,
        use_topology=True,
        maximum_quantum_edges=30,
        run_gurobi_reference=bool(USE_GUROBI and gurobi_available),
        gurobi_time_limit=GUROBI_TIME_LIMIT_MAIN,
        gurobi_mip_gap=GUROBI_MIP_GAP,
        seed=BASE_SEED + 101,
    )

    gauntlet_run, gauntlet_frame, gauntlet_output = run_and_archive(
        "03_all_constraints_gauntlet",
        gauntlet_problem,
        gauntlet_constraints,
        gauntlet_config,
    )
    assert gauntlet_run.best.breaches == 0

,value
anchor_return,0.066530
target_return,0.064534
anchor_income,0.022506
minimum_income,0.021506
anchor_cvar,0.129294
maximum_cvar,0.135759
eligible_assets,90.000000
mandatory_assets,3.000000
stress_scenarios,5.000000
cvar_scenarios,500.000000


/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)


03_all_constraints_gauntlet: best=gurobi_cardinality_miqp, objective=-0.04108150835, breaches=0, runtime=5343.626s


,method,stage,iteration,objective,runtime_seconds,feasible,breaches,max_violation,support_size,expected_return,volatility,income,turnover,best_bound,reported_mip_gap
0,scipy_extended_qp,,,-0.040659,5.723110,True,0,5.329071e-15,30,0.064956,0.084187,0.023059,0.4,,
1,feasible_initial_portfolio,initialization,,-0.039315,3.589757,True,0,1.110223e-16,20,0.064534,0.085234,0.023105,0.4,,
2,classical_enumeration,window_search,0,-0.041006,3622.593549,True,0,4.440892e-16,20,0.065147,0.084429,0.023821,0.4,,
3,xy_qaoa_subspace,window_search,0,-0.039315,0.019278,True,0,1.110223e-16,20,0.064534,0.085234,0.023105,0.4,,
4,penalty_qaoa_statevector,window_search,0,-0.041000,0.057419,True,0,4.440892e-16,20,0.065276,0.084605,0.023802,0.4,,
5,classical_enumeration,window_search,1,-0.041030,1711.268214,True,0,1.110223e-16,20,0.065303,0.084457,0.023543,0.4,,
6,xy_qaoa_subspace,window_search,1,-0.040310,0.018940,True,0,2.220446e-16,20,0.064641,0.084377,0.023243,0.4,,
7,gurobi_cardinality_miqp,global_certification,2,-0.041082,0.333819,True,0,4.440892e-16,20,0.065183,0.084282,0.023658,0.4,-0.041095,0.000331


### Optional constraint ablation

The optional ablation compares core implementation constraints, return/income/factor limits, and the complete gauntlet. It is used to explain the economic cost of stricter guardrails.

In [11]:
if RUN_CONSTRAINT_ABLATION:
    ablation_rows = []
    full_problem, full_constraints, _ = build_constraint_gauntlet(
        seed=BASE_SEED + 201
    )
    variants = {
        "core": PortfolioConstraints(
            exact_cardinality=20,
            minimum_active_weight=0.01,
            maximum_weights=np.full(full_problem.n, 0.10),
        ),
        "return_income_factor": replace(
            full_constraints,
            eligible_assets=None,
            mandatory_assets=(),
            stress_scenarios=None,
            stress_floors=None,
            scenario_returns=None,
            maximum_cvar=None,
        ),
        "all_constraints": full_constraints,
    }
    for name, constraints in variants.items():
        constraints.validate_for(full_problem)
        config = HybridConfig(
            iterations=1,
            window_size=12,
            allocation_backend="scipy",
            allocation_options={"tol": 1.0e-9, "max_iter": 8_000},
            initial_trials=300,
            initial_milp_time_limit=30.0,
            classical_tabu_iterations=60,
            classical_oracle_candidates=4,
            run_quantum=True,
            quantum=XYQAOAConfig(
                depth=1,
                shots=2048,
                optimizer_maxiter=60,
                optimizer_starts=3,
                seed=BASE_SEED + 201,
                backend="subspace",
            ),
            run_penalty_qaoa=False,
            use_topology=True,
            maximum_quantum_edges=30,
            run_gurobi_reference=False,
            seed=BASE_SEED + 201,
        )
        run = run_hybrid_optimizer(full_problem, PREFERENCES, constraints, config)
        ablation_rows.append(
            {
                "variant": name,
                "objective": run.best.objective,
                "return": run.best.metrics["expected_return"],
                "volatility": run.best.metrics["volatility"],
                "income": run.best.metrics["income"],
                "turnover": run.best.metrics["turnover"],
                "breaches": run.best.breaches,
                "runtime_seconds": run.runtime,
            }
        )
    ablation = pd.DataFrame(ablation_rows)
    display(ablation)
    ablation.to_csv(OUTPUT_ROOT / "constraint_ablation.csv", index=False)

/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/lib/python3.10/site-packages/scipy/optimize/_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)


,variant,objective,return,volatility,income,turnover,breaches,runtime_seconds
0,core,-0.037476,0.070874,0.096963,0.028199,0.4,0,9.403416
1,return_income_factor,-0.037356,0.070874,0.097034,0.028133,0.4,0,7.909080
2,all_constraints,-0.037253,0.070874,0.097196,0.028288,0.4,0,3309.120950


## 7. Freeze one portfolio state and build controlled windows

All method and backend comparisons below reuse the same universe, relaxation, exact-\(K\) initial portfolio, allocation-oracle definition, and warm start. This prevents backend effects from being confused with different adaptive windows.

In [12]:
def prepare_base_state(
    *,
    n_assets: int = FROZEN_WINDOW_ASSETS,
    cardinality: int = FROZEN_PORTFOLIO_K,
    seed: int = BASE_SEED + 301,
) -> dict[str, object]:
    problem = generate_factor_universe(
        n_assets=n_assets,
        n_groups=10,
        n_factors=12,
        seed=seed,
        current_cardinality=cardinality,
        materialize_covariance=False,
    )
    problem.max_turnover = 0.40
    constraints = PortfolioConstraints(
        exact_cardinality=cardinality,
        minimum_active_weight=0.005,
        maximum_weights=np.full(n_assets, 0.04),
    ).validate_for(problem)

    relaxation = solve_relaxation(
        problem,
        PREFERENCES,
        constraints,
        backend=ALLOCATION_BACKEND,
        solver_options={
            "tol": ALLOCATION_TOL,
            "max_iter": ALLOCATION_MAX_ITER,
        },
    )
    if not relaxation.success:
        raise RuntimeError(
            f"Relaxation failed: {relaxation.status}; "
            f"max_violation={relaxation.max_violation}"
        )

    oracle = AllocationOracle(
        problem,
        PREFERENCES,
        constraints,
        backend=ALLOCATION_BACKEND,
        solver_options={
            "tol": ALLOCATION_TOL,
            "max_iter": ALLOCATION_MAX_ITER,
        },
    )
    initial = find_feasible_initial_support(
        oracle,
        relaxation.weights,
        max_trials=500,
        seed=seed,
        milp_time_limit=60.0,
    )
    return {
        "problem": problem,
        "constraints": constraints,
        "relaxation": relaxation,
        "initial": initial,
        "communities": market_communities(problem, seed=seed),
        "seed": seed,
    }

def make_window_bundle(
    base: dict[str, object],
    width: int,
    *,
    maximum_edges: int | None = QAOA_MAX_EDGES,
) -> dict[str, object]:
    problem = base["problem"]
    constraints = base["constraints"]
    initial = base["initial"]
    relaxation = base["relaxation"]
    window = construct_change_window(
        problem,
        PREFERENCES,
        constraints,
        initial.weights,
        relaxation.weights,
        window_size=width,
        held_fraction=0.45,
        community_labels=base["communities"],
    )
    qubo = build_window_qubo(
        problem,
        PREFERENCES,
        initial.weights,
        window.indices,
        window.held_count,
        group_pressure=window.group_pressure,
        maximum_edges=maximum_edges,
    )
    return {
        **base,
        "window": window,
        "qubo": qubo,
        "initial_bits": current_window_bits(window),
    }

def new_oracle(bundle: dict[str, object]) -> AllocationOracle:
    return AllocationOracle(
        bundle["problem"],
        PREFERENCES,
        bundle["constraints"],
        backend=ALLOCATION_BACKEND,
        solver_options={
            "tol": ALLOCATION_TOL,
            "max_iter": ALLOCATION_MAX_ITER,
        },
    )

frozen_base = frozen_bundle = None
if (
    RUN_FROZEN_WINDOW_BENCHMARK
    or RUN_WIDTH_DEPTH_SWEEP
    or RUN_AER_BACKEND_COMPARISON
    or RUN_IBM_QPU
):
    frozen_base = prepare_base_state()
    frozen_bundle = make_window_bundle(frozen_base, FROZEN_WINDOW_WIDTH)
    qubo = frozen_bundle["qubo"]
    print(
        f"Frozen window: {qubo.n} qubits, {qubo.required_ones} selected, "
        f"{comb(qubo.n, qubo.required_ones):,} fixed-weight states"
    )

Frozen window: 16 qubits, 7 selected, 11,440 fixed-weight states


## 8. Exact QUBO reference and equal-candidate-budget baselines

Exact QUBO enumeration provides surrogate ground truth where practical. The final portfolio objective still comes from the allocation oracle, because the QUBO only ranks supports.

In [13]:
def exact_qubo_top_states(
    qubo,
    *,
    top_k: int = EXACT_QUBO_TOP_K,
    maximum_states: int = EXACT_QUBO_MAX_STATES,
) -> dict[str, object] | None:
    state_count = comb(qubo.n, qubo.required_ones)
    if state_count > maximum_states:
        return None
    start = time.perf_counter()
    _, bits = _fixed_weight_basis(qubo.n, qubo.required_ones)
    energies = _basis_energies(qubo, bits)
    count = min(top_k, len(energies))
    indices = np.argpartition(energies, count - 1)[:count]
    indices = indices[np.argsort(energies[indices])]
    return {
        "state_count": state_count,
        "runtime_seconds": time.perf_counter() - start,
        "best_energy": float(energies[indices[0]]),
        "worst_energy": float(np.max(energies)),
        "top_bitstrings": [bits[index].astype(int) for index in indices],
    }

def random_fixed_weight_states(
    n: int,
    required: int,
    count: int,
    seed: int,
) -> list[np.ndarray]:
    rng = np.random.default_rng(seed)
    states = []
    seen = set()
    target = min(count, comb(n, required))
    while len(states) < target:
        selected = tuple(sorted(rng.choice(n, size=required, replace=False).tolist()))
        if selected in seen:
            continue
        seen.add(selected)
        bits = np.zeros(n, dtype=int)
        bits[list(selected)] = 1
        states.append(bits)
    return states

def benchmark_row(
    method: str,
    search_runtime: float,
    allocated,
    *,
    initial_objective: float,
    sampled_energy: float | None = None,
    expected_energy: float | None = None,
    cardinality_rate: float | None = None,
    metadata: dict[str, object] | None = None,
) -> dict[str, object]:
    row = {
        "method": method,
        "search_runtime_seconds": search_runtime,
        "final_objective": allocated.best.objective,
        "objective_improvement": initial_objective - allocated.best.objective,
        "evaluated_supports": allocated.evaluated_supports,
        "feasible_supports": allocated.feasible_supports,
        "best_sampled_qubo_energy": sampled_energy,
        "expected_qubo_energy": expected_energy,
        "cardinality_rate": cardinality_rate,
    }
    if metadata:
        row.update(metadata)
    return row

frozen_benchmark = None
if RUN_FROZEN_WINDOW_BENCHMARK:
    qubo = frozen_bundle["qubo"]
    window = frozen_bundle["window"]
    initial_bits = frozen_bundle["initial_bits"]
    initial_objective = float(frozen_bundle["initial"].objective)
    rows = []

    exact = exact_qubo_top_states(qubo)
    if exact is not None:
        oracle = new_oracle(frozen_bundle)
        allocated = evaluate_bitstrings(
            "exact_qubo_top_states",
            qubo,
            [*exact["top_bitstrings"], initial_bits.copy()],
            oracle,
            window.frozen_support,
        )
        rows.append(
            benchmark_row(
                "exact_qubo_top_states",
                exact["runtime_seconds"] + allocated.runtime,
                allocated,
                initial_objective=initial_objective,
                sampled_energy=exact["best_energy"],
                metadata={"fixed_weight_state_count": exact["state_count"]},
            )
        )

    random_states = [initial_bits.copy()]
    random_states.extend(
        random_fixed_weight_states(
            qubo.n,
            qubo.required_ones,
            max(RANDOM_CANDIDATE_BUDGET - 1, 0),
            BASE_SEED + 302,
        )
    )
    oracle = new_oracle(frozen_bundle)
    start = time.perf_counter()
    random_allocated = evaluate_bitstrings(
        "random_fixed_weight",
        qubo,
        random_states,
        oracle,
        window.frozen_support,
    )
    rows.append(
        benchmark_row(
            "random_fixed_weight",
            time.perf_counter() - start,
            random_allocated,
            initial_objective=initial_objective,
            sampled_energy=min(qubo.energy(bits) for bits in random_states),
        )
    )

    oracle = new_oracle(frozen_bundle)
    lns = tabu_window_search(
        qubo,
        oracle,
        window.frozen_support,
        initial_bits,
        max_iterations=80,
        tabu_tenure=8,
        oracle_candidates_per_iteration=4,
        seed=BASE_SEED + 303,
    )
    rows.append(
        benchmark_row(
            "classical_tabu_lns",
            lns.runtime,
            lns,
            initial_objective=initial_objective,
            sampled_energy=float(lns.metadata["surrogate_best_energy"]),
        )
    )

    quantum_backends = ["subspace"]
    if RUN_AER_BACKEND_COMPARISON:
        quantum_backends.append("aer_cpu")
        if "GPU" in aer_devices:
            quantum_backends.append("aer_gpu")

    reference_angles = None
    for backend in quantum_backends:
        quantum = solve_xy_qaoa(
            qubo,
            initial_bits,
            XYQAOAConfig(
                depth=1,
                shots=QAOA_SHOTS,
                optimizer_maxiter=QAOA_OPTIMIZER_MAXITER,
                optimizer_starts=QAOA_OPTIMIZER_STARTS,
                seed=BASE_SEED + 304,
                initial_state="warm",
                mixer="ring",
                backend=backend,
                maximum_subspace_states=QAOA_MAX_SUBSPACE_STATES,
                top_candidates=QAOA_TOP_CANDIDATES,
                transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
            ),
        )
        if reference_angles is None:
            reference_angles = quantum.angles
        else:
            assert np.allclose(reference_angles, quantum.angles)

        oracle = new_oracle(frozen_bundle)
        allocated = evaluate_bitstrings(
            quantum.method,
            qubo,
            [initial_bits.copy(), *quantum.bitstrings],
            oracle,
            window.frozen_support,
        )
        rows.append(
            benchmark_row(
                quantum.method,
                quantum.runtime + allocated.runtime,
                allocated,
                initial_objective=initial_objective,
                sampled_energy=quantum.best_sampled_energy,
                expected_energy=quantum.expected_surrogate_energy,
                cardinality_rate=quantum.cardinality_feasibility_rate,
                metadata={
                    "execution_device": quantum.metadata.get("execution_device"),
                    "gpu_accelerated": quantum.metadata.get("gpu_accelerated"),
                    "transpiled_depth": quantum.metadata.get("transpiled_depth"),
                    "two_qubit_gates": quantum.metadata.get(
                        "transpiled_two_qubit_gates",
                        quantum.metadata.get("logical_two_qubit_gates"),
                    ),
                },
            )
        )

    frozen_benchmark = pd.DataFrame(rows).sort_values("final_objective")
    display(frozen_benchmark)
    frozen_benchmark.to_csv(
        OUTPUT_ROOT / "frozen_window_method_benchmark.csv",
        index=False,
    )

    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].bar(
        frozen_benchmark["method"],
        frozen_benchmark["objective_improvement"],
    )
    axes[0].set_ylabel("Improvement over warm-start portfolio")
    axes[0].tick_params(axis="x", rotation=55)
    axes[0].set_title("Validated financial improvement")

    axes[1].bar(
        frozen_benchmark["method"],
        frozen_benchmark["search_runtime_seconds"],
    )
    axes[1].set_yscale("log")
    axes[1].set_ylabel("Seconds")
    axes[1].tick_params(axis="x", rotation=55)
    axes[1].set_title("Window method time")
    figure.tight_layout()
    figure.savefig(
        OUTPUT_ROOT / "frozen_window_method_benchmark.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

,method,search_runtime_seconds,final_objective,objective_improvement,evaluated_supports,feasible_supports,best_sampled_qubo_energy,expected_qubo_energy,cardinality_rate,fixed_weight_state_count,execution_device,gpu_accelerated,transpiled_depth,two_qubit_gates
2,classical_tabu_lns,0.527391,-0.044475,0.002304,73,73,-0.063068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,random_fixed_weight,1.019241,-0.044468,0.002297,128,128,-0.063039,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,exact_qubo_top_states,0.997501,-0.044308,0.002137,129,129,-0.063068,NaN,NaN,11440.0,NaN,NaN,NaN,NaN
3,xy_qaoa_subspace,0.130637,-0.042868,0.000697,6,6,-0.057545,-0.05718,1.0,NaN,CPU,False,NaN,72.0
4,xy_qaoa_aer_cpu,0.757182,-0.042868,0.000697,5,5,-0.057545,-0.05718,1.0,NaN,CPU,False,47.0,71.0


## 9. Width/depth/trainability sweep

Increasing width enlarges the support neighborhood. Increasing depth adds expressive layers but also parameters and two-qubit operations. Depth-2 receives a larger optimizer budget than depth-1.

In [14]:
width_depth_results = None

if RUN_WIDTH_DEPTH_SWEEP:
    rows = []
    for width in WIDTH_SWEEP:
        bundle = make_window_bundle(frozen_base, width)
        qubo = bundle["qubo"]
        window = bundle["window"]
        initial_bits = bundle["initial_bits"]
        initial_objective = float(bundle["initial"].objective)
        state_count = comb(qubo.n, qubo.required_ones)

        exact = exact_qubo_top_states(qubo)
        exact_energy = None if exact is None else exact["best_energy"]

        oracle = new_oracle(bundle)
        lns = tabu_window_search(
            qubo,
            oracle,
            window.frozen_support,
            initial_bits,
            max_iterations=80,
            tabu_tenure=8,
            oracle_candidates_per_iteration=4,
            seed=BASE_SEED + 400 + width,
        )
        rows.append(
            {
                "width": width,
                "required_ones": qubo.required_ones,
                "depth": 0,
                "method": "classical_tabu_lns",
                "state_count": state_count,
                "exact_qubo_energy": exact_energy,
                "best_sampled_energy": float(
                    lns.metadata["surrogate_best_energy"]
                ),
                "energy_regret": (
                    np.nan
                    if exact_energy is None
                    else float(lns.metadata["surrogate_best_energy"]) - exact_energy
                ),
                "final_objective": lns.best.objective,
                "objective_improvement": initial_objective - lns.best.objective,
                "runtime_seconds": lns.runtime,
                "optimizer_evaluations": 0,
                "logical_two_qubit_gates": 0,
                "cardinality_rate": 1.0,
            }
        )

        depths = DEPTH_SWEEP if width in DEPTH_SWEEP_WIDTHS else [1]
        for depth in depths:
            if state_count > SWEEP_MAX_SUBSPACE_STATES:
                rows.append(
                    {
                        "width": width,
                        "required_ones": qubo.required_ones,
                        "depth": depth,
                        "method": f"xy_qaoa_{SWEEP_BACKEND}",
                        "state_count": state_count,
                        "skipped": (
                            f"{state_count:,} states exceeds "
                            f"{SWEEP_MAX_SUBSPACE_STATES:,}"
                        ),
                    }
                )
                continue

            maxiter = 100 if depth == 1 else 300
            starts = 5 if depth == 1 else 10
            quantum = solve_xy_qaoa(
                qubo,
                initial_bits,
                XYQAOAConfig(
                    depth=depth,
                    shots=QAOA_SHOTS,
                    optimizer_maxiter=maxiter,
                    optimizer_starts=starts,
                    seed=BASE_SEED + 500 + 10 * width + depth,
                    initial_state="warm",
                    mixer="ring",
                    backend=SWEEP_BACKEND,
                    maximum_subspace_states=SWEEP_MAX_SUBSPACE_STATES,
                    top_candidates=QAOA_TOP_CANDIDATES,
                    transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
                ),
            )
            oracle = new_oracle(bundle)
            allocated = evaluate_bitstrings(
                quantum.method,
                qubo,
                [initial_bits.copy(), *quantum.bitstrings],
                oracle,
                window.frozen_support,
            )
            rows.append(
                {
                    "width": width,
                    "required_ones": qubo.required_ones,
                    "depth": depth,
                    "method": quantum.method,
                    "state_count": state_count,
                    "exact_qubo_energy": exact_energy,
                    "best_sampled_energy": quantum.best_sampled_energy,
                    "energy_regret": (
                        np.nan
                        if exact_energy is None
                        else quantum.best_sampled_energy - exact_energy
                    ),
                    "final_objective": allocated.best.objective,
                    "objective_improvement": (
                        initial_objective - allocated.best.objective
                    ),
                    "runtime_seconds": quantum.runtime + allocated.runtime,
                    "optimizer_evaluations": quantum.metadata[
                        "optimizer_evaluations"
                    ],
                    "logical_two_qubit_gates": quantum.metadata[
                        "logical_two_qubit_gates"
                    ],
                    "cardinality_rate": quantum.cardinality_feasibility_rate,
                    "exact_expected_energy": quantum.metadata[
                        "exact_expected_surrogate_energy"
                    ],
                }
            )

    width_depth_results = pd.DataFrame(rows)
    display(width_depth_results)
    width_depth_results.to_csv(
        OUTPUT_ROOT / "width_depth_sweep.csv",
        index=False,
    )

    plottable = width_depth_results[
        width_depth_results["final_objective"].notna()
    ].copy()
    figure, axes = plt.subplots(1, 3, figsize=(17, 5))
    for method, group in plottable.groupby("method"):
        for depth, depth_group in group.groupby("depth"):
            label = f"{method}, p={int(depth)}"
            axes[0].plot(
                depth_group["width"],
                depth_group["objective_improvement"],
                marker="o",
                label=label,
            )
            axes[1].plot(
                depth_group["width"],
                depth_group["runtime_seconds"],
                marker="o",
                label=label,
            )

    qaoa_rows = plottable[plottable["depth"] > 0]
    for depth, group in qaoa_rows.groupby("depth"):
        axes[2].plot(
            group["width"],
            group["energy_regret"],
            marker="o",
            label=f"XY-QAOA p={int(depth)}",
        )

    axes[0].set_title("Validated portfolio improvement")
    axes[0].set_xlabel("Window width / qubits")
    axes[0].set_ylabel("Improvement over warm start")
    axes[0].legend(fontsize=8)

    axes[1].set_title("Window method runtime")
    axes[1].set_xlabel("Window width / qubits")
    axes[1].set_ylabel("Seconds")
    axes[1].set_yscale("log")
    axes[1].legend(fontsize=8)

    axes[2].set_title("Regret to exact QUBO optimum")
    axes[2].set_xlabel("Window width / qubits")
    axes[2].set_ylabel("Best sampled energy − exact minimum")
    axes[2].axhline(0.0, linewidth=1)
    axes[2].legend(fontsize=8)

    figure.tight_layout()
    figure.savefig(
        OUTPUT_ROOT / "width_depth_sweep.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

,width,required_ones,depth,method,state_count,exact_qubo_energy,best_sampled_energy,energy_regret,final_objective,objective_improvement,runtime_seconds,optimizer_evaluations,logical_two_qubit_gates,cardinality_rate,exact_expected_energy
0,8,4,0,classical_tabu_lns,70,-0.057688,-0.057688,-6.938894e-18,-0.044054,0.001883,0.261608,0,0,1.0,NaN
1,8,4,1,xy_qaoa_subspace,70,-0.057688,-0.057671,1.701114e-05,-0.043379,0.001208,0.064814,162,44,1.0,-0.053669
2,10,4,0,classical_tabu_lns,210,-0.057688,-0.057688,-6.938894e-18,-0.044324,0.002153,0.211697,0,0,1.0,NaN
3,10,4,1,xy_qaoa_subspace,210,-0.057688,-0.057643,4.469963e-05,-0.043136,0.000964,0.091786,161,60,1.0,-0.053662
4,10,4,2,xy_qaoa_subspace,210,-0.057688,-0.057659,2.866503e-05,-0.043285,0.001114,0.469621,1796,120,1.0,-0.053860
5,12,5,0,classical_tabu_lns,792,-0.057877,-0.057877,0.000000e+00,-0.044383,0.002212,0.410970,0,0,1.0,NaN
6,12,5,1,xy_qaoa_subspace,792,-0.057877,-0.057586,2.917311e-04,-0.042957,0.000785,0.113716,173,64,1.0,-0.053633
7,12,5,2,xy_qaoa_subspace,792,-0.057877,-0.057834,4.317011e-05,-0.043647,0.001476,0.773760,1957,128,1.0,-0.053955
8,14,6,0,classical_tabu_lns,3003,-0.058058,-0.058058,-6.938894e-18,-0.044308,0.002137,0.544183,0,0,1.0,NaN
9,14,6,1,xy_qaoa_subspace,3003,-0.058058,-0.057571,4.863523e-04,-0.042893,0.000722,0.144558,166,68,1.0,-0.053626


## 10. Global factor-native scaling

The scaling runner creates a fresh subprocess per trial, records peak resident memory, and keeps the portfolio cardinality and quantum window fixed while global \(n\) grows.

The current scaling script supports:

- `subspace`;
- `aer_cpu`;
- `aer_gpu`.

It intentionally does **not** use IBM Runtime. QPU studies must freeze a window and compare identical circuits; mixing queue time and changing adaptive windows into a repeated global scaling sweep would be scientifically weak.

Final study:

- 250, 500, 1,000, 2,000, 5,000, and 10,000 assets;
- three seeded repetitions;
- a fixed 16-variable quantum window;
- Gurobi certification only through 2,000 assets;
- one separate 20,000-asset stretch run.

In [ ]:
def run_scaling_study(
    *,
    label: str,
    sizes: list[int],
    repetitions: int,
    quantum_backend: str,
    gurobi: bool,
    certification_max_assets: int,
    max_turnover: float = 0.40,
    initial_milp_time_limit: float = 600.0,
    initial_trials: int = 200,
    iterations: int = 3,
    run_quantum: bool = True,
    groups: int = 10,
    factors: int = 12,
    window_size: int = 16,
) -> tuple[int, Path]:
    output = OUTPUT_ROOT / label
    command = [
        sys.executable,
        str(REPO / "scripts" / "run_hybrid_scaling.py"),
        "--sizes",
        *[str(size) for size in sizes],
        "--repetitions",
        str(repetitions),
        "--cardinality",
        str(SCALING_CARDINALITY),
        "--groups",
        str(groups),
        "--factors",
        str(factors),
        "--window-size",
        str(window_size),
        "--iterations",
        str(iterations),
        "--backend",
        ALLOCATION_BACKEND,
        "--allocation-tolerance",
        str(ALLOCATION_TOL),
        "--allocation-max-iter",
        str(ALLOCATION_MAX_ITER),
        "--minimum-active-weight",
        "0.005",
        "--maximum-weight",
        "0.04",
        "--max-turnover",
        str(max_turnover),
        "--initial-trials",
        str(initial_trials),
        "--initial-milp-time-limit",
        str(initial_milp_time_limit),
        "--classical-tabu-iterations",
        "60",
        "--classical-tabu-tenure",
        "8",
        "--classical-oracle-candidates",
        "3",
        "--enumerate-windows-up-to",
        "5000",
        "--certification-max-assets",
        str(certification_max_assets),
        "--gurobi-time-limit",
        str(SCALING_GUROBI_TIME_LIMIT),
        "--gurobi-mip-gap",
        str(GUROBI_MIP_GAP),
        "--seed",
        str(BASE_SEED),
        "--allow-failures",
        "--overwrite",
        "--output",
        str(output),
    ]

    if run_quantum:
        command.extend(
            [
                "--quantum",
                "--quantum-backend",
                quantum_backend,
                "--quantum-depth",
                "1",
                "--quantum-shots",
                str(QAOA_SHOTS),
                "--quantum-optimizer-maxiter",
                "80",
                "--quantum-optimizer-starts",
                "4",
                "--quantum-top-candidates",
                str(QAOA_TOP_CANDIDATES),
                "--maximum-subspace-states",
                str(QAOA_MAX_SUBSPACE_STATES),
                "--maximum-quantum-edges",
                str(QAOA_MAX_EDGES),
                "--transpile-optimization-level",
                str(TRANSPILE_OPTIMIZATION_LEVEL),
            ]
        )
    command.append("--gurobi" if gurobi else "--no-gurobi")

    print(" ".join(command))
    completed = subprocess.run(command, cwd=REPO, check=False)
    if not (output / "scaling_runs.csv").is_file():
        raise RuntimeError(
            f"Scaling study did not write {output / 'scaling_runs.csv'}"
        )

    plot_command = [
        sys.executable,
        str(REPO / "scripts" / "plot_hybrid_scaling.py"),
        "--input",
        str(output),
        "--output",
        str(output / "presentation_plots"),
        "--overwrite",
    ]
    subprocess.run(plot_command, cwd=REPO, check=False)
    return completed.returncode, output


scaling_smoke = None
scaling_presentation = None
scaling_100k = None

if RUN_SCALING_SMOKE:
    _, output = run_scaling_study(
        label="04_scaling_smoke",
        sizes=SCALING_SMOKE_SIZES,
        repetitions=1,
        quantum_backend=LOCAL_QUANTUM_BACKEND,
        gurobi=bool(USE_GUROBI and gurobi_available),
        certification_max_assets=2_000,
    )
    scaling_smoke = pd.read_csv(output / "scaling_runs.csv")
    display(scaling_smoke)

if RUN_SCALING_PRESENTATION:
    _, output = run_scaling_study(
        label="05_scaling_presentation",
        sizes=SCALING_PRESENTATION_SIZES,
        repetitions=SCALING_REPETITIONS,
        quantum_backend=LOCAL_QUANTUM_BACKEND,
        gurobi=bool(USE_GUROBI and gurobi_available),
        certification_max_assets=SCALING_GUROBI_MAX_ASSETS,
    )
    scaling_presentation = pd.read_csv(output / "scaling_runs.csv")
    display(scaling_presentation)

if RUN_100K_STRETCH:
    _, output = run_scaling_study(
        label="06_scaling_100000_hybrid",
        sizes=[100_000],
        repetitions=1,
        quantum_backend=LOCAL_QUANTUM_BACKEND,
        gurobi=False,
        certification_max_assets=0,
        max_turnover=2.0,
        initial_milp_time_limit=60.0,
        initial_trials=500,
        iterations=1,
        run_quantum=True,
        groups=10,
        factors=12,
        window_size=16,
    )
    scaling_100k = pd.read_csv(output / "scaling_runs.csv")
    display(scaling_100k)

/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/bin/python /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio/scripts/run_hybrid_scaling.py --sizes 250 500 1000 2000 --repetitions 1 --cardinality 50 --groups 10 --factors 12 --window-size 16 --iterations 3 --backend osqp --allocation-tolerance 1e-10 --allocation-max-iter 1000000 --minimum-active-weight 0.005 --maximum-weight 0.04 --max-turnover 0.4 --initial-trials 200 --initial-milp-time-limit 600.0 --classical-tabu-iterations 60 --classical-tabu-tenure 8 --classical-oracle-candidates 3 --enumerate-windows-up-to 5000 --certification-max-assets 2000 --gurobi-time-limit 8000 --gurobi-mip-gap 0.001 --seed 20260802 --allow-failures --overwrite --output /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio/results/presentation_benchmark_suite/04_scaling_smoke --quantum --quantum-backend subspace --quantum-depth 1 --quantum-shots 4096 --quantum-optimizer-maxiter 80 --quantum-optimizer-starts 4 --quantum-top-candidates

,study_tier,n_assets,cardinality,window_size,iterations,repetition,seed,success,best_method,best_objective,...,quantum_execution_device,quantum_gpu_verified,quantum_cardinality_rate,quantum_angle_seconds,quantum_sampler_seconds,quantum_allocation_seconds,certification_attempted,certification_completed,skipped_components,error
0,certified_reference,250,50,16,3,0,22760802,True,classical_tabu_lns,-0.045174,...,CPU,False,1.0,0.403580,0.003248,0.452694,True,True,{},NaN
1,certified_reference,500,50,16,3,0,25260802,True,gurobi_cardinality_miqp,-0.041583,...,CPU,False,1.0,0.398353,0.003185,0.564059,True,True,{},NaN
2,certified_reference,1000,50,16,3,0,30260802,True,gurobi_cardinality_miqp,-0.040734,...,CPU,False,1.0,0.392495,0.002581,0.323225,True,True,{},NaN
3,certified_reference,2000,50,16,3,0,40260802,True,gurobi_cardinality_miqp,-0.048052,...,CPU,False,1.0,0.420907,0.003020,0.768900,True,True,{},NaN


/Volumes/Local_Scratch/Andrei/miniconda3/envs/vanguard-portfolio/bin/python /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio/scripts/run_hybrid_scaling.py --sizes 250 500 1000 2000 5000 10000 20000 35000 50000 80000 --repetitions 1 --cardinality 50 --groups 10 --factors 12 --window-size 16 --iterations 3 --backend osqp --allocation-tolerance 1e-10 --allocation-max-iter 1000000 --minimum-active-weight 0.005 --maximum-weight 0.04 --max-turnover 0.4 --initial-trials 200 --initial-milp-time-limit 600.0 --classical-tabu-iterations 60 --classical-tabu-tenure 8 --classical-oracle-candidates 3 --enumerate-windows-up-to 5000 --certification-max-assets 2000 --gurobi-time-limit 8000 --gurobi-mip-gap 0.001 --seed 20260802 --allow-failures --overwrite --output /mnt/Home/apomorov/Wiser/vanguard-multi-asset-portfolio/results/presentation_benchmark_suite/05_scaling_presentation --quantum --quantum-backend subspace --quantum-depth 1 --quantum-shots 4096 --quantum-optimizer-maxiter 80 --quantum-o

## 11. IBM QPU hardware-validation experiment

### One environment for CPU, GPU, and QPU

The pinned `full` extra uses Qiskit 2.5.1, Aer 0.17.2, and IBM Runtime
0.48.0. On Linux x86_64 it installs `qiskit-aer-gpu-cu11`, which contains
both CPU and GPU simulation and runs on newer CUDA 12/13-capable NVIDIA
drivers. Other platforms receive CPU Aer. Exactly one Aer distribution must
exist in the environment.

For a clean environment:

```bash
python -m pip install -e ".[full]"
python -m pip check
python scripts/install_environment.py --verify-only
```

To repair an environment that previously mixed CPU Aer, CUDA-12 Aer, or
Qiskit 1.4 packages:

```bash
python scripts/install_environment.py
```

Restart the Jupyter kernel after installation. The preflight prints
`sys.executable`; it must be the interpreter where the project was installed.

### One-time IBM account setup

Do not put a token in this notebook or repository. If no default account has
been saved, run this once in a private interactive cell and then delete that
cell:

```python
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=getpass("IBM Quantum API token: "),
    overwrite=True,
    set_as_default=True,
)
```

The experiment checkpoints the CSV after every CPU reference and QPU job.
With `QPU_RESUME=True`, rerunning the cell skips successful jobs and retries
only failed or missing jobs.

### What this experiment can prove

- the circuit executed on a named IBM backend;
- the raw hardware fixed-cardinality rate;
- the number of useful unique fixed-weight supports;
- financial feasibility after exact reallocation;
- objective improvement, or lack of improvement, relative to the warm start;
- hardware depth and two-qubit operation count;
- QPU usage time and queue-inclusive wall time.

It cannot by itself prove QPU speed advantage, quantum advantage, a
full-universe QPU solve, or superiority over Gurobi/LNS. Angles are optimized
classically in the fixed-weight subspace and transferred to IBM hardware.


In [ ]:
qpu_results = None
qpu_errors = []
qpu_output_path = OUTPUT_ROOT / "ibm_qpu_hardware_validation.csv"


def _qpu_row_key(row: dict[str, object]) -> tuple[str, int, int, int]:
    return (
        str(row["backend"]),
        int(row["width"]),
        int(row["depth"]),
        int(row["repetition"]),
    )


def _save_qpu_row(
    rows: list[dict[str, object]],
    row: dict[str, object],
) -> None:
    key = _qpu_row_key(row)
    for index, existing in enumerate(rows):
        if _qpu_row_key(existing) == key:
            rows[index] = row
            break
    else:
        rows.append(row)
    pd.DataFrame(rows).to_csv(qpu_output_path, index=False)


if RUN_IBM_QPU:
    from importlib import metadata as package_metadata

    try:
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    except Exception as exc:
        raise RuntimeError(
            "IBM Runtime is unavailable in this notebook kernel. Run "
            f"{sys.executable} -m pip install -e '.[full]', restart the "
            "kernel, and rerun the environment preflight."
        ) from exc

    quantum_versions = {
        name: package_metadata.version(name)
        for name in ("qiskit", "qiskit-ibm-runtime")
    }
    if quantum_versions != {
        "qiskit": "2.5.1",
        "qiskit-ibm-runtime": "0.48.0",
    }:
        raise RuntimeError(
            f"Unexpected quantum package versions: {quantum_versions}. "
            "Run scripts/install_environment.py and restart this kernel."
        )
    assert SamplerV2 is not None

    try:
        service = QiskitRuntimeService()
    except Exception as exc:
        raise RuntimeError(
            "IBM Runtime is installed, but no usable default account was "
            "found. Save the account once with "
            "QiskitRuntimeService.save_account(channel="
            "'ibm_quantum_platform', token=..., set_as_default=True), "
            "then rerun this cell."
        ) from exc

    required_qubits = max(QPU_WIDTHS)
    try:
        if IBM_BACKEND_NAME:
            selected_backend = service.backend(IBM_BACKEND_NAME)
        else:
            selected_backend = service.least_busy(
                operational=True,
                simulator=False,
                min_num_qubits=required_qubits,
                use_fractional_gates=False,
            )
    except Exception as exc:
        raise RuntimeError(
            f"No accessible operational IBM backend has {required_qubits} "
            f"qubits for the active account: {exc}"
        ) from exc

    backend_name_value = getattr(selected_backend, "name", "")
    selected_backend_name = (
        str(backend_name_value())
        if callable(backend_name_value)
        else str(backend_name_value)
    )
    backend_qubits = int(getattr(selected_backend, "num_qubits", 0))
    backend_status = selected_backend.status()
    if not bool(getattr(backend_status, "operational", False)):
        raise RuntimeError(
            f"Selected backend {selected_backend_name!r} is not operational: "
            f"{getattr(backend_status, 'status_msg', 'unknown status')}"
        )
    if backend_qubits < required_qubits:
        raise RuntimeError(
            f"Selected backend {selected_backend_name!r} has "
            f"{backend_qubits} qubits; {required_qubits} are required."
        )

    print("IBM Runtime channel:", getattr(service, "channel", "unknown"))
    print("IBM backend:", selected_backend_name)
    print("Backend qubits:", backend_qubits)
    print("Pending jobs at selection:", getattr(backend_status, "pending_jobs", "unknown"))

    rows: list[dict[str, object]] = []
    if QPU_RESUME and qpu_output_path.is_file():
        previous = pd.read_csv(qpu_output_path)
        rows = previous.to_dict(orient="records")
        for row in rows:
            if "status" not in row or pd.isna(row["status"]):
                row["status"] = "success"
        print(f"Loaded {len(rows)} checkpoint rows from {qpu_output_path}")

    completed = {
        _qpu_row_key(row)
        for row in rows
        if str(row.get("status", "")) == "success"
    }

    for width in QPU_WIDTHS:
        bundle = make_window_bundle(
            frozen_base,
            width,
            maximum_edges=QPU_MAX_EDGES,
        )
        qubo = bundle["qubo"]
        window = bundle["window"]
        initial_bits = bundle["initial_bits"]
        state_count = comb(qubo.n, qubo.required_ones)
        exact = exact_qubo_top_states(qubo)
        exact_energy = None if exact is None else exact["best_energy"]

        for depth in QPU_DEPTHS:
            cpu_config = XYQAOAConfig(
                depth=depth,
                shots=QPU_SHOTS,
                optimizer_maxiter=QPU_OPTIMIZER_MAXITER,
                optimizer_starts=QPU_OPTIMIZER_STARTS,
                seed=BASE_SEED + 700 + 10 * width + depth,
                initial_state="warm",
                mixer="ring",
                backend="subspace",
                maximum_subspace_states=max(
                    QAOA_MAX_SUBSPACE_STATES,
                    state_count,
                ),
                top_candidates=QAOA_TOP_CANDIDATES,
                transpile_optimization_level=TRANSPILE_OPTIMIZATION_LEVEL,
            )
            ideal = solve_xy_qaoa(qubo, initial_bits, cpu_config)
            cpu_key = ("exact_subspace_cpu", width, depth, 0)
            if cpu_key not in completed:
                ideal_oracle = new_oracle(bundle)
                ideal_allocated = evaluate_bitstrings(
                    ideal.method,
                    qubo,
                    [initial_bits.copy(), *ideal.bitstrings],
                    ideal_oracle,
                    window.frozen_support,
                )
                _save_qpu_row(
                    rows,
                    {
                        "status": "success",
                        "error": "",
                        "backend": "exact_subspace_cpu",
                        "width": width,
                        "required_ones": qubo.required_ones,
                        "depth": depth,
                        "repetition": 0,
                        "shots": QPU_SHOTS,
                        "job_id": "",
                        "calibration_timestamp": "",
                        "state_count": state_count,
                        "exact_qubo_energy": exact_energy,
                        "best_sampled_energy": ideal.best_sampled_energy,
                        "energy_regret": (
                            np.nan
                            if exact_energy is None
                            else ideal.best_sampled_energy - exact_energy
                        ),
                        "cardinality_rate": ideal.cardinality_feasibility_rate,
                        "unique_bitstrings": ideal.metadata[
                            "unique_sampled_bitstrings"
                        ],
                        "validated_objective": ideal_allocated.best.objective,
                        "objective_improvement": (
                            float(bundle["initial"].objective)
                            - float(ideal_allocated.best.objective)
                        ),
                        "transpiled_depth": np.nan,
                        "two_qubit_gates": ideal.metadata[
                            "logical_two_qubit_gates"
                        ],
                        "qpu_usage_seconds": np.nan,
                        "qpu_wall_seconds": np.nan,
                    },
                )
                completed.add(cpu_key)

            for repetition in range(1, QPU_REPETITIONS + 1):
                hardware_key = (
                    selected_backend_name,
                    width,
                    depth,
                    repetition,
                )
                if hardware_key in completed:
                    print("Skipping completed QPU job:", hardware_key)
                    continue

                try:
                    qpu_config = replace(
                        cpu_config,
                        backend="ibm_runtime",
                        ibm_backend=selected_backend_name,
                    )
                    qpu = solve_xy_qaoa(qubo, initial_bits, qpu_config)
                    if not np.allclose(ideal.angles, qpu.angles):
                        raise RuntimeError(
                            "CPU and QPU paths did not reuse identical angles"
                        )

                    qpu_oracle = new_oracle(bundle)
                    qpu_allocated = evaluate_bitstrings(
                        qpu.method,
                        qubo,
                        [initial_bits.copy(), *qpu.bitstrings],
                        qpu_oracle,
                        window.frozen_support,
                    )
                    row = {
                        "status": "success",
                        "error": "",
                        "backend": selected_backend_name,
                        "width": width,
                        "required_ones": qubo.required_ones,
                        "depth": depth,
                        "repetition": repetition,
                        "shots": QPU_SHOTS,
                        "job_id": qpu.metadata.get("job_id", ""),
                        "calibration_timestamp": qpu.metadata.get(
                            "backend_calibration_timestamp",
                            "",
                        ),
                        "state_count": state_count,
                        "exact_qubo_energy": exact_energy,
                        "best_sampled_energy": qpu.best_sampled_energy,
                        "energy_regret": (
                            np.nan
                            if exact_energy is None
                            else qpu.best_sampled_energy - exact_energy
                        ),
                        "cardinality_rate": qpu.cardinality_feasibility_rate,
                        "unique_bitstrings": qpu.metadata[
                            "unique_sampled_bitstrings"
                        ],
                        "validated_objective": qpu_allocated.best.objective,
                        "objective_improvement": (
                            float(bundle["initial"].objective)
                            - float(qpu_allocated.best.objective)
                        ),
                        "transpiled_depth": qpu.metadata.get(
                            "transpiled_depth"
                        ),
                        "two_qubit_gates": qpu.metadata.get(
                            "transpiled_two_qubit_gates"
                        ),
                        "qpu_usage_seconds": qpu.metadata.get(
                            "qpu_usage_seconds"
                        ),
                        "qpu_wall_seconds": qpu.metadata.get(
                            "qpu_wall_seconds"
                        ),
                    }
                    _save_qpu_row(rows, row)
                    completed.add(hardware_key)
                    print(
                        "Completed",
                        hardware_key,
                        "job_id=",
                        row["job_id"],
                    )
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}"
                    qpu_errors.append(
                        {
                            "backend": selected_backend_name,
                            "width": width,
                            "depth": depth,
                            "repetition": repetition,
                            "error": error,
                        }
                    )
                    _save_qpu_row(
                        rows,
                        {
                            "status": "failed",
                            "error": error,
                            "backend": selected_backend_name,
                            "width": width,
                            "required_ones": qubo.required_ones,
                            "depth": depth,
                            "repetition": repetition,
                            "shots": QPU_SHOTS,
                            "job_id": "",
                            "calibration_timestamp": "",
                            "state_count": state_count,
                            "exact_qubo_energy": exact_energy,
                        },
                    )
                    print("QPU job failed:", hardware_key, error)
                    if QPU_FAIL_FAST:
                        raise

    qpu_results = pd.DataFrame(rows)
    if not qpu_results.empty:
        qpu_results = qpu_results.sort_values(
            ["backend", "width", "depth", "repetition"],
            ignore_index=True,
        )
        qpu_results.to_csv(qpu_output_path, index=False)
    display(qpu_results)
    if qpu_errors:
        display(pd.DataFrame(qpu_errors))

    successful = qpu_results[
        qpu_results["status"].astype(str) == "success"
    ].copy()
    hardware = successful[
        successful["backend"] != "exact_subspace_cpu"
    ].copy()

    if hardware.empty:
        print("No successful IBM hardware rows are available to plot yet.")
    else:
        figure, axes = plt.subplots(2, 2, figsize=(15, 10))
        for (backend, depth), group in successful.groupby(
            ["backend", "depth"]
        ):
            aggregate = group.groupby(
                "width",
                as_index=False,
            ).median(numeric_only=True)
            label = f"{backend}, p={int(depth)}"
            axes[0, 0].plot(
                aggregate["width"],
                aggregate["cardinality_rate"],
                marker="o",
                label=label,
            )
            axes[0, 1].plot(
                aggregate["width"],
                aggregate["objective_improvement"],
                marker="o",
                label=label,
            )

        for depth, group in hardware.groupby("depth"):
            aggregate = group.groupby(
                "width",
                as_index=False,
            ).median(numeric_only=True)
            axes[1, 0].plot(
                aggregate["width"],
                aggregate["two_qubit_gates"],
                marker="o",
                label=f"p={int(depth)}",
            )
            axes[1, 1].plot(
                aggregate["width"],
                aggregate["qpu_wall_seconds"],
                marker="o",
                label=f"wall, p={int(depth)}",
            )
            if aggregate["qpu_usage_seconds"].notna().any():
                axes[1, 1].plot(
                    aggregate["width"],
                    aggregate["qpu_usage_seconds"],
                    marker="s",
                    linestyle="--",
                    label=f"QPU usage, p={int(depth)}",
                )

        axes[0, 0].set_title("Raw fixed-cardinality shot rate")
        axes[0, 0].set_ylabel("Rate")
        axes[0, 0].set_ylim(0.0, 1.02)
        axes[0, 1].set_title("Validated portfolio improvement")
        axes[0, 1].set_ylabel("Improvement over warm start")
        axes[1, 0].set_title("Hardware circuit burden")
        axes[1, 0].set_ylabel("Transpiled two-qubit gates")
        axes[1, 1].set_title("QPU time")
        axes[1, 1].set_ylabel("Seconds")
        for axis in axes.flat:
            axis.set_xlabel("Qubits / window width")
            axis.legend(fontsize=8)
            axis.grid(alpha=0.25)

        figure.tight_layout()
        for suffix in ("png", "pdf"):
            figure.savefig(
                OUTPUT_ROOT / f"ibm_qpu_hardware_validation.{suffix}",
                dpi=220,
                bbox_inches="tight",
            )
        plt.show()


## 12. Optional appendix — legacy alternative quantum formulations

These older equal-lot QAOA/PCE and VQE/PCE scripts do not share the canonical
continuous-weight hybrid model or identical constraint semantics. They are
disabled by default and should be run only as a separately labeled appendix.


In [ ]:
if RUN_LEGACY_ALTERNATIVE_MODELS:
    legacy_commands = [
        [
            sys.executable,
            str(REPO / "scripts" / "run_quantum.py"),
            "--n-lots",
            "20",
            "--n-restarts",
            "5",
            "--reps",
            "8",
            "--seed",
            str(BASE_SEED),
        ],
        [
            sys.executable,
            str(REPO / "scripts" / "run_quantum_vqe.py"),
        ],
        [
            sys.executable,
            str(REPO / "scripts" / "run_vqe_pce.py"),
        ],
        [
            sys.executable,
            str(REPO / "scripts" / "compare_all.py"),
            str(
                REPO
                / "data"
                / "synthetic"
                / "synthetic_universe.json"
            ),
        ],
    ]
    for command in legacy_commands:
        print("\nRUNNING:", " ".join(command))
        subprocess.run(command, cwd=REPO, check=False)

## 13. Optional Copilot launch

Run in a terminal for the live demonstration:

```bash
streamlit run src/vanguard_portfolio/copilot_app.py
```

Recommended demo:

- 50–100 assets;
- exact cardinality 15–20;
- balanced preferences;
- one advanced guardrail at a time;
- a fixed seed;
- the independent validation panel.

Do not enable every advanced guardrail for the live demo unless that combination has already been tested. Use the all-constraints gauntlet as the archived proof.

In [ ]:
def copilot_command() -> list[str]:
    return [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(
            REPO
            / "src"
            / "vanguard_portfolio"
            / "copilot_app.py"
        ),
    ]

print("Copilot command:")
print(" ".join(copilot_command()))

## 14. Generate evidence-backed presentation claims

This cell writes a Markdown file containing only claims supported by artifacts that actually exist. It intentionally distinguishes:

- Gurobi-certified gaps;
- gaps to the continuous relaxation;
- Aer GPU verification;
- QPU hardware validation;
- strict zero-breach results.

In [ ]:
claim_lines = [
    "# Evidence-backed presentation claims",
    "",
    f"- Repository commit: `{environment['commit']}` "
    f"on branch `{environment['branch']}`.",
]


if continuous_backend_crosscheck is not None:
    successful_backends = continuous_backend_crosscheck[
        continuous_backend_crosscheck["success"] == True
    ]
    if len(successful_backends) >= 2:
        spread = (
            successful_backends["objective"].max()
            - successful_backends["objective"].min()
        )
        claim_lines.append(
            f"- {len(successful_backends)} independent continuous-QP "
            f"backends agreed within an objective spread of {spread:.3e}."
        )

if tiny_run is not None:
    claim_lines.append(
        f"- Tiny exact-certification case completed with "
        f"{tiny_run.best.breaches} hard-constraint breaches."
    )
    tiny_gurobi = tiny_frame[
        tiny_frame["method"] == "gurobi_cardinality_miqp"
    ]
    tiny_enum = tiny_frame[
        tiny_frame["method"] == "classical_enumeration"
    ]
    if not tiny_gurobi.empty and not tiny_enum.empty:
        spread = abs(
            float(tiny_gurobi.iloc[-1]["objective"])
            - float(tiny_enum.iloc[-1]["objective"])
        )
        claim_lines.append(
            f"- Tiny enumeration and Gurobi objectives differ by "
            f"{spread:.3e}."
        )

if main_run is not None:
    claim_lines.append(
        f"- The 100-asset canonical hybrid case returned "
        f"{np.count_nonzero(main_run.best.weights > 1e-8)} "
        f"holdings with {main_run.best.breaches} "
        "hard-constraint breaches."
    )
    gurobi_rows = main_frame[
        main_frame["method"] == "gurobi_cardinality_miqp"
    ]
    if not gurobi_rows.empty:
        row = gurobi_rows.iloc[-1]
        gap = row.get("reported_mip_gap")
        if pd.notna(gap):
            claim_lines.append(
                f"- Gurobi reported a MIP gap of "
                f"{float(gap):.4%} for the 100-asset reference."
            )


if gurobi_start_study is not None:
    fastest = gurobi_start_study.loc[
        gurobi_start_study["total_seconds"].idxmin()
    ]
    claim_lines.append(
        f"- In the controlled Gurobi start study, "
        f"`{fastest['start']}` was fastest at "
        f"{float(fastest['total_seconds']):.3f} s; "
        "cold and warm starts are reported separately."
    )

if backtest_robustness_summary is not None:
    claim_lines.append(
        f"- Out-of-sample behavior was evaluated across "
        f"{BACKTEST_ROBUSTNESS_PATHS} independent synthetic paths, "
        "with medians and 10th–90th percentile ranges reported."
    )

if gauntlet_run is not None:
    claim_lines.append(
        "- One constraint-gauntlet case simultaneously enforced "
        "exact cardinality, minimum/maximum positions, eligibility, "
        "mandatory assets, group limits, turnover, target return, "
        "minimum income, factor bands, stress floors, and empirical CVaR."
    )
    claim_lines.append(
        f"- The all-constraints portfolio had "
        f"{gauntlet_run.best.breaches} independently recomputed breaches."
    )

if frozen_benchmark is not None:
    best_row = frozen_benchmark.iloc[0]
    claim_lines.append(
        f"- On the frozen {FROZEN_WINDOW_WIDTH}-asset window, "
        f"the best tested method was `{best_row['method']}` "
        f"with validated improvement "
        f"{float(best_row['objective_improvement']):.6g} "
        "over the warm start."
    )
    gpu_rows = frozen_benchmark[
        frozen_benchmark["method"].astype(str).str.contains(
            "aer_gpu"
        )
    ]
    if (
        not gpu_rows.empty
        and bool(gpu_rows.iloc[0].get("gpu_accelerated", False))
    ):
        claim_lines.append(
            "- Aer execution metadata verified GPU sampling "
            "for the frozen-window circuit."
        )

scaling_candidates = []
for frame in (
    scaling_smoke,
    scaling_presentation,
    scaling_100k,
):
    if frame is not None:
        scaling_candidates.append(frame)

if scaling_candidates:
    scaling_all = pd.concat(
        scaling_candidates,
        ignore_index=True,
    )
    successful = scaling_all[
        scaling_all["success"] == True
    ].copy()
    if not successful.empty:
        maximum_size = int(successful["n_assets"].max())
        maximum_rows = successful[
            successful["n_assets"] == maximum_size
        ]
        claim_lines.append(
            f"- The largest successful factor-native universe "
            f"contained {maximum_size:,} assets."
        )
        claim_lines.append(
            f"- At {maximum_size:,} assets, median time to the "
            f"first valid portfolio was "
            f"{maximum_rows['time_to_first_valid_seconds'].median():.3f} s "
            f"and median complete hybrid search time was "
            f"{maximum_rows['search_end_to_end_seconds'].median():.3f} s."
        )
        claim_lines.append(
            f"- Across recorded scaling trials, "
            f"{int((successful['breaches'] == 0).sum())}/"
            f"{len(successful)} successful runs had zero breaches."
        )

if qpu_results is not None and not qpu_results.empty:
    hardware = qpu_results[
        (qpu_results["backend"] != "exact_subspace_cpu")
        & (qpu_results["status"].astype(str) == "success")
        & qpu_results["job_id"].fillna("").astype(str).ne("")
    ].copy()
    if not hardware.empty:
        claim_lines.append(
            f"- IBM Runtime hardware evidence contains "
            f"{hardware['job_id'].nunique()} distinct successful job IDs."
        )
        claim_lines.append(
            f"- Median raw QPU fixed-cardinality rate was "
            f"{hardware['cardinality_rate'].median():.2%}."
        )
        feasible_improvements = hardware[
            "objective_improvement"
        ].dropna()
        if not feasible_improvements.empty:
            claim_lines.append(
                f"- Best QPU-generated validated improvement over "
                f"the warm start was "
                f"{feasible_improvements.max():.6g}."
            )
    else:
        claim_lines.append(
            "- No successful IBM hardware job with a non-empty job ID was "
            "available; no QPU-performance claim was generated."
        )

claim_lines.extend(
    [
        "",
        "## Required interpretation limits",
        "",
        "- Global asset count is not QPU width; the QPU receives "
        "only the adaptive window.",
        "- A continuous-relaxation gap is not a certified "
        "mixed-integer optimality gap.",
        "- Cardinality preservation demonstrates circuit correctness, "
        "not quantum advantage.",
        "- QPU queue-inclusive wall time and QPU usage time must be "
        "reported separately.",
        "- Synthetic backtests are demonstrations, not investment forecasts.",
    ]
)

claims_path = OUTPUT_ROOT / "winning_claims.md"
claims_path.write_text(
    "\n".join(claim_lines) + "\n",
    encoding="utf-8",
)
print(claims_path.read_text(encoding="utf-8"))

## 15. Recommended final presentation evidence

### Main deck

1. Architecture and mathematical model.
2. Tiny exact-certification result.
3. Main 100-asset objective-quality / runtime / Gurobi panel.
4. All-constraints gauntlet with a large **0 breaches** callout.
5. Top buys/sells and support-overlap figures.
6. Global scaling: first-valid time, total runtime, memory, and oracle calls.
7. Quantum-window study: width/depth quality, exact-QUBO regret, and circuit burden.
8. IBM QPU hardware-validation panel, if completed.
9. Copilot live demo.

### Appendix

- penalty-QAOA versus XY-QAOA cardinality;
- complete constraint table;
- correlation/community heatmap;
- circuit depth and two-qubit gates;
- backtest paths and multi-seed summaries;
- legacy VQE/PCE studies;
- package and hardware environment;
- limitations and the no-quantum-advantage statement.